# Battery Q-Learning Notebook

This notebook uses `battery_q_learning.jl` as the main implementation.
It runs training on 2016-2024 and testing on 2025, then previews outputs.

In [1]:
using Pkg
required_pkgs = ["CSV", "DataFrames", "XLSX", "Statistics", "Dates", "Random"]
for pkg in required_pkgs
    if Base.find_package(pkg) === nothing
        Pkg.add(pkg)
    end
end

In [2]:
using CSV, DataFrames
include("battery_q_learning.jl")

In [3]:
params = ModelParams(
    train_years = collect(2016:2024),
    test_years = [2025],
    consolidated_data_file = "data/hub_north_2021_2026.xlsx",
    consolidated_sheet_name = "hub_north_all",
    dt_hours = 0.25,
)

# You can modify these bins directly here:
# params = ModelParams(n_price_bins=21, n_soc_bins=15, n_soh_bins=5, n_life_bins=5)

ModelParams([2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024], [2025], "hub_north", "hub_north_2021_2026.xlsx", "hub_north_all", 0.937, 250.0, 0.25, 9223372036854775807, 1000.0, 0.0, 1000.0, 1.0, 0.85, 15.0, 7300.0, 2.0547945205479456e-5, 0.1, 15.0, 21, 15, 5, 5, 50, 0.15, 0.03, 0.995, 1.0, 0.05, 0.9, 50.0, 300.0, true, 50.0, 42)

In [4]:
result = run_pipeline(params)
result.summary

=== Dataset ===
Train years: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024], rows: 315648
Test years : [2025], rows: 35040

=== Training ===
Episodes: 50
Episode 1 epsilon/alpha : 1.0 / 0.15
Last episode epsilon/alpha: 0.05 / 0.03
Last-episode train profit: 3.498371338e7
Last-episode cycles used : 5155.911

=== Test Result ===
Total profit: 5.67038353e6
Final SOH  : 0.987441
Cycles used: 611.211
Action counts (charge/idle/discharge): 10100/9970/14970
Saved timeline: q_learning_2025_timeline.csv
Saved training log: training_episode_log.csv


(total_profit = 5.6703835306944605e6, final_soh = 0.9874408659263414, cycles_used = 611.2111915822239, charge_steps = 10100, idle_steps = 9970, discharge_steps = 14970)

In [5]:
timeline = CSV.read("results/q_learning_2025_timeline.csv", DataFrame)
first(timeline, 30)

Row,timestamp,price,action,soc_mwh,soh,cycles_used,step_cash,cumulative_profit
,DateTime,Float64,String15,Float64,Float64,Float64,Float64,Float64
1,2025-01-01T00:00:00,17.9,idle,1000.0,1.0,0.0,0.0,0.0
2,2025-01-01T00:15:00,17.98,idle,1000.0,1.0,0.0,0.0,0.0
3,2025-01-01T00:30:00,18.21,idle,1000.0,1.0,0.0,0.0,0.0
4,2025-01-01T00:45:00,18.22,idle,1000.0,1.0,0.0,0.0,0.0
5,2025-01-01T01:00:00,17.97,idle,1000.0,1.0,0.0,0.0,0.0
6,2025-01-01T01:15:00,18.06,idle,1000.0,1.0,0.0,0.0,0.0
7,2025-01-01T01:30:00,17.36,discharge,937.5,0.999999,0.03125,1050.27,1050.27
8,2025-01-01T01:45:00,17.36,discharge,875.0,0.999999,0.0625,1050.27,2100.53
9,2025-01-01T02:00:00,17.3,discharge,812.5,0.999998,0.09375,1046.64,3147.17


In [6]:
# Optional: quick bin search on validation split (fit years < validation_year, test on validation_year)
bin_results = quick_bin_search(
    params;
    price_candidates = [21, 31, 41],
    soc_candidates = [15, 21],
    soh_candidates = [5, 7],
    life_candidates = [5, 7],
    tuning_episodes = 6,
    validation_year = 2024,
)
first(bin_results, 10)

Row,n_price_bins,n_soc_bins,n_soh_bins,n_life_bins,val_profit,val_final_soh,val_cycles
,Int64,Int64,Int64,Int64,Float64,Float64,Float64
1,21,15,7,7,1.42803e6,0.993124,334.609
2,21,15,5,7,1.1793e6,0.994162,284.12
3,21,15,7,5,1.1793e6,0.994162,284.12
4,31,21,5,7,4.78241e5,0.993033,339.084
5,31,21,7,5,4.78241e5,0.993033,339.084
6,31,21,7,7,3.61025e5,0.989526,509.752
7,31,15,5,7,3.09588e5,0.999459,26.3332
8,31,15,7,5,3.09588e5,0.999459,26.3332
9,21,21,7,7,2.89158e5,0.989746,499.014
